# 🌿 Sanjeevani: Life Beyond Cancer — Multimodal PANAS-20 Affective Computing Engine
### Production Multimodal Diagnostic & Psychosocial Triage Notebook (SRS Module 2, 3, 4 & Section 8)

This notebook deploys the **Multimodal Affective AI Engine** developed for **Sanjeevani – Life Beyond Cancer**, an Indian cancer care NGO supporting over 1.5 million patients across 33 government hospital centers in 15 states.

### 🔬 Research Grounding
- **Monk Prayogshala (Dec 2018)**: Validated the 20-item Positive and Negative Affect Scale (PANAS) on Indian cancer patients (PA $\alpha = 0.79$, NA $\alpha = 0.81$). Demonstrates a statistically significant drop in Negative Affect from $20.29$ to $14.97$ ($p < 0.0001$).
- **College of Social Work, Nirmala Niketan (2017)**: 250-patient study across 7 centers (TMH Mumbai, GCRI Ahmedabad, SMS Jaipur). Oncologists affirmed the urgent need for automated psychosocial triage in OPDs.
- **Srjan & Satori Clinical Interventions (2025)**: Significant reductions in pain ($6.76 \to 3.42$), anxiety ($7.32 \to 3.56$), and a $50\%$ drop in Mini-MAC Helplessness-Hopelessness.

### 🚀 How to Run in Google Colab
1. In Google Colab, select **Runtime → Change runtime type → T4 GPU** (optional, CPU also supported).
2. Click **Runtime → Run all** (~45 seconds).
3. Open the public `gradio.live` link generated at the bottom.

In [ ]:
# Cell 1: Environment Installation (~25 seconds)
!pip install -q gradio transformers datasets librosa soundfile accelerate torchaudio
print('✓ Multimodal environment ready!')

In [ ]:
# Cell 2: PANAS-20 Taxonomy, Research Anchors & Knowledge Base Ingestion
import os
import re
import json
from datetime import datetime
import numpy as np
import pandas as pd
import librosa
import gradio as gr

# ------------------------------------------------------------------------------
# 1. The 20 PANAS Affect Dimensions (Watson et al., 1988 & Monk Prayogshala 2018)
# ------------------------------------------------------------------------------
PANAS_PA_ITEMS = [
    "interested", "excited", "strong", "enthusiastic", "proud",
    "alert", "inspired", "determined", "attentive", "active"
]

PANAS_NA_ITEMS = [
    "distressed", "upset", "guilty", "scared", "hostile",
    "irritable", "ashamed", "nervous", "jittery", "afraid"
]

ALL_PANAS_ITEMS = PANAS_PA_ITEMS + PANAS_NA_ITEMS

PANAS_META = {
    "interested": {"label": "Interested / Inquisitive (इच्छुक)", "type": "PA", "color": "#059669", "desc": "Seeking clarity, curious about self-care"},
    "excited": {"label": "Excited / Joyful (उत्साहित)", "type": "PA", "color": "#10b981", "desc": "Anticipatory relief, milestone celebration"},
    "strong": {"label": "Strong / Resilient (बलवान / सहनशील)", "type": "PA", "color": "#047857", "desc": "Inner physical and emotional fortitude"},
    "enthusiastic": {"label": "Enthusiastic (उत्साहयुक्त)", "type": "PA", "color": "#14b8a6", "desc": "Motivated to engage in yoga/therapy"},
    "proud": {"label": "Proud / Victorious (गर्व)", "type": "PA", "color": "#0d9488", "desc": "Cancer victor dignity, overcoming stigma"},
    "alert": {"label": "Alert / Mindful (सतर्क)", "type": "PA", "color": "#0284c7", "desc": "Vigilant monitoring of symptoms and blood counts"},
    "inspired": {"label": "Inspired / Uplifted (प्रेरित)", "type": "PA", "color": "#6366f1", "desc": "Motivated by peer survivors and mentors"},
    "determined": {"label": "Determined / Resolute (अवधारित / दृढ़)", "type": "PA", "color": "#4338ca", "desc": "Unwavering fighting spirit, will to heal"},
    "attentive": {"label": "Attentive / Disciplined (सचेत)", "type": "PA", "color": "#3b82f6", "desc": "Adherent to clinical protocols and self-care"},
    "active": {"label": "Active / Engaged (सक्रिय)", "type": "PA", "color": "#16a34a", "desc": "Physical mobility, walking, daily yogasana"},
    "distressed": {"label": "Distressed / Overwhelmed (व्यथित)", "type": "NA", "color": "#dc2626", "desc": "General psychological suffering and trauma"},
    "upset": {"label": "Upset / Agitated (परेशान होना)", "type": "NA", "color": "#ea580c", "desc": "Troubled by nausea, vomiting, or setbacks"},
    "guilty": {"label": "Guilty / Burdened (दोषी)", "type": "NA", "color": "#b45309", "desc": "Self-blame for medical bills draining family savings"},
    "scared": {"label": "Scared / Panicked (भयभीत)", "type": "NA", "color": "#d97706", "desc": "Acute terror of scan, biopsy needle, or recurrence"},
    "hostile": {"label": "Hostile / Bitter (शत्रुतापूर्ण / विरुद्ध)", "type": "NA", "color": "#991b1b", "desc": "Resentment: 'Why me? Why did God punish me?'"},
    "irritable": {"label": "Irritable / Snapping (चिड़चिड़ा)", "type": "NA", "color": "#c2410c", "desc": "Short temper from steroids, chemo brain, or insomnia"},
    "ashamed": {"label": "Ashamed / Stigmatized (शर्मिंदा)", "type": "NA", "color": "#475569", "desc": "Alopecia, mastectomy scar, or stoma bag shame"},
    "nervous": {"label": "Nervous / Apprehensive (बेचैन)", "type": "NA", "color": "#ca8a04", "desc": "Waiting room jitters, anticipatory anxiety"},
    "jittery": {"label": "Jittery / Trembling (कमजोर नसें / कंपन)", "type": "NA", "color": "#e11d48", "desc": "Physical tremors, heart racing, acute panic"},
    "afraid": {"label": "Afraid / Mortality Dread (डरा हुआ)", "type": "NA", "color": "#7f1d1d", "desc": "Existential dread of death or leaving children"}
}

# Multilingual Affective Lexicon Anchors
PANAS_LEXICON = {
    "interested": ["interested", "curious", "want to know", "learn", "details chahiye", "janna hai", "इच्छुक", "उत्सुक", "जिज्ञासु"],
    "excited": ["excited", "thrilled", "joyful", "celebrating", "great news", "khush", "khushi", "उत्साहित", "प्रसन्न"],
    "strong": ["strong", "resilient", "tough", "strength", "himmat", "taqat", "majboot", "बलवान", "सहनशील", "मजबूत"],
    "enthusiastic": ["enthusiastic", "eager", "motivated", "ready", "shuru karna", "उत्साहयुक्त", "उमंग"],
    "proud": ["proud", "pride", "dignity", "cancer victor", "garv", "shaan", "गर्व", "गौरवान्वित"],
    "alert": ["alert", "vigilant", "monitoring", "tracking", "satark", "dhyan rakh raha", "सतर्क", "सचेत"],
    "inspired": ["inspired", "inspirational", "uplifted", "hopeful", "motivation mila", "himmat badhi", "प्रेरित", "हौसला"],
    "determined": ["determined", "resolute", "will fight", "wont give up", "dridh", "haar nahi manunga", "अवधारित", "दृढ़"],
    "attentive": ["attentive", "focused", "mindful", "listening", "dhyan se", "follow kar raha", "सचेत", "एकाग्र"],
    "active": ["active", "walking", "moving", "yoga", "chalta firta", "energetic", "सक्रिय", "टहलना"],
    "distressed": ["distressed", "overwhelmed", "suffering", "agony", "unbearable", "toota hua", "bardasht nahi", "व्यथित", "पीड़ा"],
    "upset": ["upset", "troubled", "nausea", "vomiting", "miserable", "pareshan", "ulti", "tabiyat kharab", "परेशान"],
    "guilty": ["guilty", "burden", "draining savings", "self blame", "bojh", "meri wajah se", "दोषी", "अपराधबोध"],
    "scared": ["scared", "terrified", "panic", "needle fear", "darr", "dar lag raha", "kaanp raha", "भयभीत", "डरा हुआ"],
    "hostile": ["hostile", "angry", "hate", "unfair", "gussa", "krodh", "kismat kharab", "शत्रुतापूर्ण", "गुस्सा"],
    "irritable": ["irritable", "annoyed", "short tempered", "mood swings", "chidchida", "chidchidapan", "चिड़चिड़ा"],
    "ashamed": ["ashamed", "embarrassed", "hair loss", "disfigured", "sharam", "baal jhad gaye", "mirror nahi dekhta", "शर्मिंदा"],
    "nervous": ["nervous", "anxious", "restless", "waiting room", "bechain", "ghabrahat", "bechaini", "बेचैन", "घबराहट"],
    "jittery": ["jittery", "shaking", "trembling", "palpitations", "haath kaanp rahe", "shivering", "थरथराना", "कंपन"],
    "afraid": ["afraid", "fear of death", "dying", "leaving kids", "maut ka darr", "marne se darr", "डरा हुआ", "मौत"]
}

NEGATIONS = ["not", "never", "no", "dont", "without", "nahi", "nhi", "na", "mat", "bina", "नहीं", "न", "बिना"]
CRISIS_PHRASES = ["mar jaana chahta", "suicide", "jeena nahi chahta", "kill myself", "end my life", "unbearable pain", "sab khatam"]

SANJEEVANI_KB = {
    "helpline": "8691000801 / 8691000802",
    "programs": {
        "cansahyogi": "CanSahyogi: Flagship psycho-social patient navigation & emotional counselling across 33 government hospital centers.",
        "satori": "Satori: Holistic healing combining therapeutic yoga, pranayama, yoganidra, CBT, and emotional freedom technique (EFT).",
        "canahaar": "CanAhaar: Nutritional support bags (almonds, roasted chana, rajgira, jowar flour) and herbal infusions for chemo patients.",
        "srjan": "Srjan: 21-session comprehensive cancer recovery program; clinically proven to cut pain (6.76->3.42) and anxiety (7.32->3.56).",
        "cansaarthi": "CanSaarthi: 6-month intensive certification course training cancer victors as professional oncology caregivers.",
        "aham_sanjeevani": "Aham Sanjeevani: 104 annual sessions of continued survivor-led support, lymphedema care, and Reiki healing."
    }
}
print('✓ PANAS-20 clinical scale and Sanjeevani Knowledge Base initialized!')

In [ ]:
# Cell 3: Multimodal Diagnostic Engine (Acoustic Prosody + Multilingual Text)

def extract_text_panas(text):
    if not text or not text.strip():
        return None
    text_lower = text.lower()
    words = re.findall(r'\b[\w\u0900-\u097f]+\b', text_lower)

    scores = {item: 1.0 for item in ALL_PANAS_ITEMS}
    has_esc = any(p in text_lower for p in CRISIS_PHRASES)

    for item, terms in PANAS_LEXICON.items():
        matched = 0
        for t in terms:
            if ' ' in t and t in text_lower:
                matched += 1
            else:
                for i, w in enumerate(words):
                    if w == t:
                        # Check negation window (previous 3 tokens)
                        window = words[max(0, i-3):i]
                        if not any(neg in window for neg in NEGATIONS):
                            matched += 1
        if matched > 0:
            scores[item] = round(1.0 + min(4.0, matched * 1.5), 2)

    if has_esc:
        scores["distressed"] = max(scores["distressed"], 4.9)
        scores["afraid"] = max(scores["afraid"], 4.8)
        scores["scared"] = max(scores["scared"], 4.7)

    pa_sum = round(sum(scores[k] for k in PANAS_PA_ITEMS), 2)
    na_sum = round(sum(scores[k] for k in PANAS_NA_ITEMS), 2)

    return {
        "scores": scores,
        "pa_sum": pa_sum,
        "na_sum": na_sum,
        "has_escalation": has_esc
    }

def extract_audio_panas(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return None
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
    except Exception:
        return None

    if y is None or len(y) == 0:
        return None

    rms = float(np.sqrt(np.mean(y**2)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))

    scores = {item: 1.0 for item in ALL_PANAS_ITEMS}

    # High vocal tension & tremors (Panic, Jitter, Anxiety)
    if zcr > 0.065:
        stress = min(4.0, (zcr / 0.065) * 2.0)
        scores["jittery"] += stress * 1.4
        scores["nervous"] += stress * 1.2
        scores["scared"] += stress * 1.0

    # Flat / Hypophonic vocal affect (Chemo exhaustion, shock, grief)
    if rms < 0.04:
        blunt = min(4.0, (0.04 / max(0.005, rms)) * 1.5)
        scores["distressed"] += blunt * 1.2
        scores["upset"] += blunt * 1.0
        scores["afraid"] += blunt * 0.8
    elif rms > 0.09 and zcr < 0.06:
        vigor = min(4.0, (rms / 0.09) * 1.8)
        scores["strong"] += vigor * 1.3
        scores["determined"] += vigor * 1.1
        scores["active"] += vigor * 1.0

    scores = {k: max(1.0, min(5.0, round(v, 2))) for k, v in scores.items()}
    pa_sum = round(sum(scores[k] for k in PANAS_PA_ITEMS), 2)
    na_sum = round(sum(scores[k] for k in PANAS_NA_ITEMS), 2)

    return {
        "scores": scores,
        "pa_sum": pa_sum,
        "na_sum": na_sum,
        "biomarkers": {"rms": round(rms, 4), "zcr": round(zcr, 4)}
    }

def fuse_multimodal_panas(text_res, audio_res):
    if text_res and audio_res:
        mode = "Multimodal (Linguistic + Speech Acoustics)"
        fused = {}
        for itm in ALL_PANAS_ITEMS:
            fused[itm] = round(0.60 * text_res["scores"][itm] + 0.40 * audio_res["scores"][itm], 2)
        pa_sum = round(sum(fused[k] for k in PANAS_PA_ITEMS), 2)
        na_sum = round(sum(fused[k] for k in PANAS_NA_ITEMS), 2)

        # Multimodal Alignment & Masked Distress Detection
        t_pa, t_na = text_res["pa_sum"], text_res["na_sum"]
        a_na = audio_res["na_sum"]
        alignment = None

        if t_pa > 25.0 and t_na < 16.0 and (a_na >= 20.0 or audio_res["biomarkers"]["zcr"] > 0.07):
            alignment = {
                "badge": "⚠️ Clinical Alert: Masked Distress Detected",
                "badge_class": "warning",
                "title": "Verbal Positivity Contradicts Vocal Distress",
                "detail": f"Patient's written words express optimism/calm (PA: {t_pa}/50), but vocal acoustic tremor reveals acute physiological tension (Acoustic NA: {a_na}/50, ZCR: {audio_res['biomarkers']['zcr']}). Frequent indicator of emotional suppression / masking in cancer care."
            }
        elif t_na >= 22.0 and a_na < 15.0 and audio_res["biomarkers"]["rms"] < 0.035:
            alignment = {
                "badge": "ℹ️ Clinical Notice: Blunted Vocal Demeanor",
                "badge_class": "info",
                "title": "Cognitive Distress with Flat Vocal Demeanor",
                "detail": f"Patient expresses distress verbally, but vocal power is severely depressed and monotone (RMS: {audio_res['biomarkers']['rms']}), typical of chemotherapy fatigue or shock."
            }
        else:
            alignment = {
                "badge": "✅ High Modality Concordance",
                "badge_class": "success",
                "title": "Harmonious Cross-Modal Expression",
                "detail": f"Both verbal semantics (NA: {t_na}) and acoustic prosody (NA: {a_na}) synchronously reflect genuine affective state."
            }

        return {"mode": mode, "scores": fused, "pa_sum": pa_sum, "na_sum": na_sum, "alignment": alignment, "has_escalation": text_res["has_escalation"]}
    elif text_res:
        return {"mode": "Text-Only (Linguistic)", "scores": text_res["scores"], "pa_sum": text_res["pa_sum"], "na_sum": text_res["na_sum"], "alignment": None, "has_escalation": text_res["has_escalation"]}
    elif audio_res:
        return {"mode": "Voice-Only (Acoustic Prosody)", "scores": audio_res["scores"], "pa_sum": audio_res["pa_sum"], "na_sum": audio_res["na_sum"], "alignment": None, "has_escalation": False}
    else:
        return {"mode": "None", "scores": {i: 1.0 for i in ALL_PANAS_ITEMS}, "pa_sum": 10.0, "na_sum": 10.0, "alignment": None, "has_escalation": False}

print('✓ Multimodal diagnostic pipeline ready!')

In [ ]:
# Cell 4: Emotional Distress Score (EDS 0-100) & Grounded RAG Generator (SRS Section 8)

def compute_eds_triage(fused_result, conversation_history=None):
    if conversation_history is None:
        conversation_history = []

    pa_sum = fused_result["pa_sum"]
    na_sum = fused_result["na_sum"]
    has_esc = fused_result.get("has_escalation", False)
    scores = fused_result["scores"]

    max_na = max([scores.get(k, 1.0) for k in PANAS_NA_ITEMS], default=1.0)
    norm_na_sum = max(0.0, min(100.0, ((na_sum - 10.0) / 30.0) * 100.0))
    norm_peak = max(0.0, min(100.0, ((max_na - 1.0) / 4.0) * 100.0))
    i_na = 0.60 * norm_na_sum + 0.40 * norm_peak

    total_affect = pa_sum + na_sum
    s_neg = max(0.0, min(100.0, ((na_sum - 10.0) / (total_affect - 19.0)) * 100.0)) if total_affect > 20.0 else 0.0
    base_distress = 0.65 * i_na + 0.35 * s_neg

    # Multi-turn history
    distress_turns = sum(1 for t in conversation_history if t.get("na_sum", 10) >= 18.0)
    r_repeated = 0.0 if distress_turns == 0 else (12.0 if distress_turns == 1 else 20.0)
    p_esc = 35.0 if has_esc else 0.0

    raw_eds = base_distress + r_repeated + p_esc
    if has_esc:
        raw_eds = max(raw_eds, 85.0)

    eds = round(max(0.0, min(100.0, raw_eds)), 1)

    # 4 Triage Bands
    if eds <= 30.0:
        level, category, badge = "GREEN", "Normal conversation", "🟢"
        action = "No action; standard chatbot flow continues."
    elif eds <= 60.0:
        level, category, badge = "YELLOW", "Emotional support recommended", "🟡"
        action = "Chatbot offers counselling-resource information alongside its answer."
    elif eds <= 80.0:
        level, category, badge = "ORANGE", "Counsellor attention recommended", "🟠"
        action = "Conversation is added to the escalation queue for review within standard SLA."
    else:
        level, category, badge = "RED", "Priority human review", "🔴"
        action = "Conversation flagged for immediate counsellor attention; AI provides crisis de-escalation."

    return {"eds": eds, "level": level, "category": category, "badge": badge, "action": action}

def generate_sanjeevani_response(text, triage):
    helpline = SANJEEVANI_KB["helpline"]
    level = triage["level"]

    if level == "RED":
        return (f"Hum samajh sakte hain ki aap is waqt bohot zyada pain aur acute distress me hain. "
                f"Aap bilkul akele nahi hain, Sanjeevani team aapke sath khadi hai. "
                f"Humne aapka message immediate counsellor review ke liye flag kar diya hai. "
                f"Aap direct Sanjeevani helpline: {helpline} par call ya WhatsApp kar sakte hain.")
    elif level == "ORANGE":
        return (f"Treatment ke dauran aisi anxiety aur nervousness hona bilkul natural hai. "
                f"Sanjeevani ka CanSahyogi program isi psycho-social support aur hospital guidance ke liye banaya gaya hai. "
                f"Aapki baat humne counsellor queue me add kar di hai. Helpline: {helpline}.")
    elif level == "YELLOW":
        return (f"Aapke sawaal ke liye shukriya. Sanjeevani ke Satori Holistic Healing program aur CanAhaar nutrition bags "
                f"chemotherapy ke side effects kam karne me bohot madad karte hain. "
                f"Counselling aur information ke liye contact: {helpline}.")
    else:
        return (f"Sanjeevani – Life Beyond Cancer me aapka swagat hai. "
                f"Hum cancer prevention, OPD navigation, nutrition aur survivor support ke 14 programs chalate hain. "
                f"Kisi bhi inquiry ke liye sampark karein: {helpline}.")

print('✓ EDS tracker and grounded response generator ready!')

In [ ]:
# Cell 5: Gradio Interactive Multimodal Demo Interface

def run_full_diagnosis(text, audio_path):
    if not text and not audio_path:
        return "<div style='color:red; text-align:center;'>Please provide text, voice audio, or both.</div>", "", ""

    text_res = extract_text_panas(text)
    audio_res = extract_audio_panas(audio_path)
    fused = fuse_multimodal_panas(text_res, audio_res)
    triage = compute_eds_triage(fused)
    reply = generate_sanjeevani_response(text, triage)

    # Color mapping
    t_color = "#16a34a" if triage["level"] == "GREEN" else ("#b45309" if triage["level"] == "YELLOW" else ("#c2410c" if triage["level"] == "ORANGE" else "#dc2626"))

    # Gauge HTML
    gauge_html = f"""
    <div style="background: white; border: 1.5px solid #cbd5e1; border-top: 4px solid {t_color}; border-radius: 12px; padding: 20px; text-align: center;">
        <div style="font-size: 11px; font-weight: 800; text-transform: uppercase; color: #64748b;">SRS Section 8 Emotional Distress Score</div>
        <div style="font-size: 42px; font-weight: 800; color: {t_color}; line-height: 1.1;">{triage['eds']} <span style="font-size: 18px; color: #94a3b8;">/ 100</span></div>
        <div style="display: inline-block; padding: 4px 12px; border-radius: 6px; font-size: 13px; font-weight: 800; margin-top: 6px; border: 1px solid {t_color}; color: {t_color};">
            {triage['badge']} Level {triage['level']}: {triage['category']}
        </div>
        <div style="font-size: 13px; color: #334155; margin-top: 8px;"><b>System Action:</b> {triage['action']}</div>
        <div style="font-size: 12px; color: #047857; margin-top: 6px;"><b>Modality Engine:</b> {fused['mode']}</div>
    </div>
    """

    # Alignment HTML
    align_html = ""
    if fused.get("alignment"):
        a = fused["alignment"]
        align_html = f"""
        <div style="background: white; border: 1.5px solid #cbd5e1; border-radius: 12px; padding: 18px; margin-top: 14px;">
            <div style="font-size: 12px; font-weight: 800; text-transform: uppercase; color: #475569;">{a['badge']}</div>
            <div style="font-size: 16px; font-weight: 800; color: #0f172a; margin: 4px 0;">{a['title']}</div>
            <div style="font-size: 13.5px; color: #334155; line-height: 1.5;">{a['detail']}</div>
        </div>
        """

    # Spectrum HTML
    sorted_items = sorted(fused["scores"].items(), key=lambda x: x[1], reverse=True)
    bars = ""
    for item, sc in sorted_items[:6]:
        meta = PANAS_META.get(item, {})
        pct = int((sc / 5.0) * 100)
        bars += f"""
        <div style="margin-bottom: 8px;">
            <div style="display: flex; justify-content: space-between; font-size: 12.5px; font-weight: 700;">
                <span>{meta.get('label', item)}</span>
                <span style="color: {meta.get('color', '#333')};">{sc}/5.0 ({pct}%)</span>
            </div>
            <div style="height: 7px; background: #e2e8f0; border-radius: 4px; overflow: hidden;">
                <div style="width: {pct}%; height: 100%; background: {meta.get('color', '#333')};"></div>
            </div>
        </div>
        """

    spec_html = f"""
    <div style="background: white; border: 1.5px solid #cbd5e1; border-radius: 12px; padding: 18px; margin-top: 14px;">
        <div style="font-size: 14px; font-weight: 800; color: #0f172a; margin-bottom: 10px;">Top Activated PANAS-20 Affect Dimensions</div>
        {bars}
        <div style="background: #f0fdf4; border: 1.5px solid #86efac; border-radius: 8px; padding: 12px; margin-top: 12px;">
            <div style="font-size: 11px; font-weight: 800; color: #065f46; text-transform: uppercase;">Grounded Empathetic Reply</div>
            <div style="font-size: 13.5px; color: #0f172a; margin-top: 4px;">{reply}</div>
        </div>
    </div>
    """

    return gauge_html, align_html, spec_html

with gr.Blocks(title="Sanjeevani Multimodal Affective AI") as demo:
    gr.HTML("""
    <div style="background: linear-gradient(135deg, #064e3b 0%, #047857 70%, #059669 100%); border-radius: 12px; padding: 22px 28px; color: white; margin-bottom: 18px;">
        <div style="font-size: 11px; font-weight: 800; text-transform: uppercase; letter-spacing: 0.5px; opacity: 0.9;">CanSupport AI • Sanjeevani Life Beyond Cancer</div>
        <div style="font-size: 24px; font-weight: 800; margin: 4px 0;">Multimodal PANAS-20 Oncology Affective Computing</div>
        <div style="font-size: 14px; opacity: 0.95;">Processes English, Hindi, and Hinglish text + speech audio to detect masked distress and automate clinical triage.</div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            text_in = gr.Textbox(label="Patient Written Statement (English / Hindi / Hinglish)", placeholder="E.g., Kal mera PET scan hai, bohot darr lag raha hai...", lines=4)
            audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Patient Voice Recording (Speech Acoustics)")
            btn = gr.Button("Analyze Multimodal Affective State", variant="primary")
        with gr.Column(scale=1):
            out_gauge = gr.HTML()
            out_align = gr.HTML()
            out_spec = gr.HTML()

    btn.click(fn=run_full_diagnosis, inputs=[text_in, audio_in], outputs=[out_gauge, out_align, out_spec])

demo.launch(share=True, debug=True)